# 1.13 第二次 backward 為什麼梯度變大？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**同一 loss 做兩次新的 forward/backward，grad 會一樣嗎？

PyTorch 的 grad 預設會累加，而不是每次覆蓋。這能刻意合併 micro-batch，但忘記清零就改了本來的更新規則。

**把直覺寫成數學：**$g_{stored}\leftarrow g_{stored}+g_{new}$；zero_grad 才把上一輪梯度清掉。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
w = nn.Parameter(torch.tensor(2.0))
(w**2).backward()
first = w.grad.clone()
(w**2).backward()
second = w.grad.clone()
w.grad = None
(w**2).backward()
print(first, second, w.grad)
assert torch.allclose(second, 2 * first)

**如何讀結果：**每次使用新的 forward 計算圖，沒有重用已釋放的圖。

**只改一件事：**只移除清零那一行，預測第三次的梯度。
